# PSP8 — Coleta e ETL em nuvem (Kaggle → Google Colab → Google Drive)

**Etapa 2:** validar persistência de um modelo de dados *flat*, com camadas bronze (arquivo de origem preservado), prata (atributos tipados) e evidências de qualidade e integridade. Este notebook **não altera** o notebook original do SSD.

A base é de projetos de infraestrutura em geral, não de data centers. Este fluxo ainda **não** executa análises de viabilidade nem comprova desempenho em situações reais.


## 1. Preparar o Colab
1. No Colab, execute todas as células em sequência, com acesso à internet.
2. Autorize a montagem do Google Drive quando solicitado.
3. Depois da execução, confira os CSVs nas pastas `bronze` e `prata`, a auditoria, o manifesto e os hashes.
4. Guarde capturas reais da pasta do Drive e das mensagens de validação para as evidências acadêmicas. **Não** publique credenciais nem arquivos privados.


In [ ]:
%pip install -q kagglehub pandas


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Carregar o código versionado
O Colab baixa a implementação do pipeline a partir da branch de trabalho. Para uma entrega reprodutível final, a URL deve ser fixada em um commit validado antes de mesclar na `main`.


In [ ]:
from pathlib import Path
import urllib.request

URL_SCRIPT = (
    'https://raw.githubusercontent.com/ricardopimentelandre/'
    'MVP-Viabilidade-Nova-Unidade-Dados/psp8-etapa-2-pipeline/'
    'scripts/pipeline_etl_drive.py'
)
arquivo_script = Path('/content/pipeline_etl_drive.py')
urllib.request.urlretrieve(URL_SCRIPT, arquivo_script)
print('Código de ETL obtido:', arquivo_script)


In [ ]:
import sys
sys.path.insert(0, '/content')
from pipeline_etl_drive import executar_pipeline

DESTINO_DRIVE = Path('/content/drive/MyDrive/UnB/PSP8/MVP_Viabilidade_Nova_Unidade')
resultado = executar_pipeline(DESTINO_DRIVE)


## 3. Remontar o Drive e verificar a persistência
O Drive montado pode servir leituras do cache local. Por isso, antes da conferência, o Drive é **desmontado (com flush) e remontado**; em seguida `verificar_persistencia` relê os arquivos gravados, recalcula SHA-256, tamanhos, contagens e refaz a auditoria, comparando tudo com o manifesto gravado no Drive. Se algo divergir, a célula falha. Isso **não substitui** a captura da pasta no navegador.

In [ ]:
from google.colab import drive
drive.flush_and_unmount()
print('Drive desmontado (flush concluído).')

In [ ]:
drive.mount('/content/drive', force_remount=True)

In [ ]:
from pipeline_etl_drive import verificar_persistencia
verificacao = verificar_persistencia(Path(resultado['pasta_execucao']), remontado=True)
print('Resultado:', verificacao['resultado'])

## 4. Resumo para as evidências
Capture a tela desta saída e, no navegador, a pasta `execucoes/<id>/` do Google Drive com `bronze/`, `prata/` e `evidencias/` (inclui `verificacao_persistencia.json`).

In [ ]:
import json
manifesto = resultado['manifesto']
auditoria = resultado['auditoria']
print('Estado:', manifesto['status'])
print('Local no Drive:', manifesto['destino_execucao'])
print('Fonte:', manifesto['dataset_handle'], '| versão baixada:', manifesto['versao_kaggle_baixada'], '| licença:', manifesto['licenca_dos_dados'])
print('Coletado em (UTC):', manifesto['coletado_em_utc'], '| executado em (UTC):', manifesto['execucao_utc'])
print('Registros bronze/prata:', manifesto['registros_bronze'], manifesto['registros_prata'])
print('SHA-256 fonte :', manifesto['sha256_fonte'])
print('SHA-256 bronze:', manifesto['sha256_bronze'])
print('SHA-256 prata :', manifesto['sha256_prata'])
print('Prata idêntica à bronze (bytes):', manifesto['prata_identica_a_bronze_byte_a_byte'])
print('Hash confere com a v1 documentada:', manifesto['sha256_fonte_confere_com_v1_documentada'])
print('Violações do domínio esperado:', auditoria['resumo']['total_violacoes_dominio_esperado'])
print('Outliers IQR (informativo):', auditoria['resumo']['total_outliers_iqr_informativo'])

In [ ]:
import pandas as pd
from IPython.display import display
prata_do_drive = pd.read_csv(resultado['caminho_prata'])
assert len(prata_do_drive) == manifesto['registros_prata']
assert len(prata_do_drive.columns) == 13
display(prata_do_drive.head(3))
resumo = pd.DataFrame(auditoria['atributos']).T
display(resumo[['tipo_pandas', 'nulos', 'valores_unicos', 'fora_do_dominio_esperado']])

## 5. Limitações e próximo ponto de controle
- O CSV original é preservado em `bronze`; a camada `prata` preserva as 13 colunas e tipa os campos numéricos. Não há exclusão de registros nesta etapa.
- A licença (CC0) e a versão (1) vêm da API do Kaggle e estão no manifesto; reconfirme a licença na página do dataset antes da entrega.
- Nenhuma transformação altera o conteúdo: a prata é cópia validada e tipada da bronze (ver `transformacoes_prata` no manifesto).
- A auditoria identifica indícios de problemas, mas não corrige nem garante acurácia semântica.
- Os dados **não** devem ser considerados persistidos no Drive até este notebook executar com sucesso e a pasta ser verificada.
- A análise das perguntas P1–P4 e o short paper são etapas posteriores, sujeitas à validação.